In [1]:
import torch
import torch.nn as nn
import math

In [2]:
# Input Embeddings 

class InputEmbeddings(nn.Module):
    def __init__(self,vocab_size,d_model):
        super().__init__() #initializing parent -> nn.Module
        self.embedding = nn.Embedding(vocab_size,d_model)
        self.d_model = d_model
    def forward(self,x):
        return self.embedding(x) * math.sqrt(self.d_model) #scaling -> to keep the embedding magintude in a suitable range relative to positional encoding -> thus improving stability

In [3]:
# Positional Encoding -> during parallelization each token doesn't know its relative distance from one another , this facilitates this 

class PositionalEncoding(nn.Module):
    def __init__(self,d_model,max_len=5000): #maxlen -> max seq the model can handle at once
        super().__init__()

        pe = torch.zeroes(max_len,d_model)

        position = torch.arrange(0,max_len).unsqueeze(1) #unsqueeze -> adds a new dim -> converts pos to column vector which further facilitates broadcasting

        div_term = torch.exp(torch.arrange(0,d_model,2)*(-math.log(10000.0)/d_model))

        pe[:,0::2] = torch.sin(position * div_term)
        pe[:,1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)

        self.register_buffer("pe",pe) # buffer is stored inside the model but not trainable

    def forward(self,x): # adds all this positional info to the embedding
        return x + self.pe[:,x.size(1)]
        

In [5]:
# Self Attention

class MultiHeadAttention(nn.Module):
    def __init__(self,d_model,num_heads):
        super().__init__()

        assert d_mdel % num_heads ==0 #if condition is true it continues else raise error ; num_heads -> intuitive experts
        self.d_model = d_model
        self.num_heads = num_heads
        seld.d_k = d_model//num_heads
        self.W_q = nn.Linear(d_model, d_model) #querry vector
        self.W_k = nn.Linear(d_model, d_model) #key vector
        self.W_v = nn.Linear(d_model, d_model) #values vector

        self.W_o = nn.Linear(d_model, d_model)


In [6]:
def split_heads(self, x):
    batch_size, seq_len , d_model = x.shape #batch size , no of words in each sentence , dimention of vector representing each word

    x = x.view(
        batch_size,
        seq_len,
        self.num_heads,
        self.d_k)
    return x.transpose(1,2)


In [7]:
def scaled_dot_product_attention(self,Q,K,V,mask=None):  #scaled-> divides by underoot of dimension of key vector ; masking -> to prevent cheating/data leage will be used later in decoder
    scores = torch.matmul(Q.K.transpose(-2,-1)) #dot product -> gives similarity score
    scores = scores/math.sqrt(self.d_k)

    if mask is not None:
        scores = masked_fill(mask==0,dim=-1) # applying masking for decoder -> prevents data leakage

    attention = torch.softmax(scores,dim=-1)
    output = torch.matmul(attention,V)
    return output
        

In [10]:
def forward (self,q,k,v,mask=None):
    batch_size = q.shape[0]
    Q = self.W_q(q)
    K=self.W_k(k)
    V=self.W_v(v)

    Q=self.split_heads(Q)
    K=self.split_heads(K)
    v=self.split_heads(V)

    output = self.scaled_dot_product_attention(Q,K,V,mask)
    output = output.transpose(1,2).contiguous()
    output=output.view(batch_size,-1,self.d_model)
    return self.W_o(output)

In [12]:
class LayerNormalization(nn.Module):
    def __init__(self,features,eps=1e-6):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(features))
        self.beta = nn.Parameter(torch.zeros(features))
        self.eps = eps
    def forward(self,x):
        mean = x.mean(-1,keepdim=True)
        std = x.std(-1,keepdim=True)

        return self.gamma * (x-mean)/(std+self.eps) + self.beta
        

In [16]:
class FeedForward(nn.Module): #processes the info attention has gathered
    def __init__(self,d_model,d_ff):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(d_model,d_ff),
            nn.ReLU(), #introduces non linearity
            nn.Linear(d_ff,d_model)
        )
    def forward(self,x):
        return self.net(x)

In [17]:
class AddNorm(nn.Module):
    def __init__(self,d_model,dropout):
        super().__init__()

        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self,x,sublayer):
        return self.norm(
            x + self.dropout(sublayer)
        )
        

In [18]:
class EncoderBlock(nn.Module):
    def __init(
        self,
        d_model,
        num_heads,
        d_ff,
        dropout):
        super().__init__()
        self.attention = MultiHeadAttention(
            d_model,
            num_heads
        )
        self.ffn = FeedForward(
            d_model,
            d_ff
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
        self.dropout = nn.Dropout(d_model)

    def forward(self,x,mask=None):
        attn = self.attention(x,x,x,mask)
        x = self.norm1(
            x + self.dropout(attn)
        )
        ffn = self.ffn(x)
        x = self.norm2(
            x + self.dropout(ffn)
        )
        return x

In [20]:
class Encoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model,
        num_heads,
        d_ff,
        num_layers,
        dropout):
        super().__init__()
        self.embedding = InputEmbedding(vocab_size,d_model)
        self.position = PositionalEncoding(d_model)
        self.layers = nn.ModuleList([
            EncoderBlock(d_model,num_heads,d_ff,dropout)for _ in range(num_layers)
        ])

    def forward(self,x,mask=None):
        x = self.embedding(x)
        x = self.position(x)
        for layer in self.layers:
            x = layer(x,mask)
        return x
        

In [22]:
class DecoderBlock(nn.Module):
    def __init__(
        self,
        d_model,
        num_heads,
        d_ff,
        dropout):
        super().__init__()
        # Masked Self Attentiion 
        self.selfl_attention = MultiHeadAttention(
            d_model,
            num_heads
        )
        #Cross Attention
        self.cross_attention = MultiHeadAttention(
            d_model,
            num_heads
        )
        self.ffn = Feedforward(
            d_model,d_ff
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)
    def forward(
        self,
        x
        encoder_output,
        src_mask = None,
        tgt_mask = None
    )
        #Masked Self Attention
        attn = self.self_attention(
            x,
            x,
            x,
            tgt_mask
        )
        x = self.norm2(
            x + self.dropout(attn)
        )
        


        